# Weekly Store Sales Forecasting

## Project Overview

This project forecasts **weekly total store sales** using a combination of statistical,
machine-learning, and AutoML approaches. We generate a realistic synthetic dataset
spanning ~3 years of weekly sales with trend, yearly seasonality, holiday spikes, and noise.

**Models used**: Naive baselines, LazyPredict (tabular lag-feature), FLAML AutoML,
and StatsForecast (AutoARIMA, AutoETS, SeasonalNaive).

**Forecast horizon**: 8 weeks ahead.

## Learning Objectives

1. Understand weekly time-series patterns (trend, yearly seasonality)
2. Build naive and seasonal-naive baselines for comparison
3. Engineer lag and rolling features for tabular ML on time series
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML for automated model selection
6. Use StatsForecast for classical statistical forecasting
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical weekly sales data, predict the total sales for the next **8 weeks**.
Accurate weekly forecasts help retailers plan staffing, inventory, and promotions.

## Why This Project Matters

Weekly forecasting is a common cadence for retail operations. Unlike daily data,
weekly aggregation smooths out day-of-week effects but retains seasonal and trend signals.
Getting this right drives better supply-chain and workforce decisions.

## Dataset Overview

We use a **synthetic dataset** with the following characteristics:
- ~156 weeks (~3 years) of weekly sales
- Upward trend component
- Yearly seasonality (52-week cycle)
- Holiday bumps (weeks near Christmas / New Year)
- Random noise

| Column | Description |
|--------|-------------|
| `date` | Week start date (Monday) |
| `sales` | Total weekly sales ($) |

## Dataset Source and License Notes

This dataset is **synthetically generated** for educational purposes.
No external data download is required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_WEEKS = 160          # ~3 years of weekly data
FORECAST_HORIZON = 8   # predict 8 weeks ahead
VAL_WEEKS = 8
TEST_WEEKS = 8
SEASON_LENGTH = 52     # yearly seasonality
np.random.seed(SEED)
print(f"Config: {N_WEEKS} weeks, horizon={FORECAST_HORIZON}, season={SEASON_LENGTH}")

Config: 160 weeks, horizon=8, season=52


## Dataset Generation

In [4]:
dates = pd.date_range(start='2021-01-04', periods=N_WEEKS, freq='W-MON')
t = np.arange(N_WEEKS)

trend = 50000 + 200 * t
yearly = 8000 * np.sin(2 * np.pi * t / 52)

# Holiday bumps (weeks 50-52 each year)
holiday = np.zeros(N_WEEKS)
for i in range(N_WEEKS):
    week_of_year = dates[i].isocalendar()[1]
    if week_of_year >= 50 or week_of_year <= 1:
        holiday[i] = 15000

noise = np.random.normal(0, 3000, N_WEEKS)
sales = trend + yearly + holiday + noise
sales = np.maximum(sales, 5000)

df = pd.DataFrame({'date': dates, 'sales': sales})
df['date'] = pd.to_datetime(df['date'])
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

Dataset shape: (160, 2)
Date range: 2021-01-04 00:00:00 to 2024-01-22 00:00:00


,date,sales
0,2021-01-04,66490.142459
1,2021-01-11,50749.500539
2,2021-01-18,54257.590929
3,2021-01-25,58005.928666
4,2021-02-01,53815.325252


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values found"
assert df['sales'].min() > 0, "Negative sales found"
assert df['date'].is_monotonic_increasing, "Dates not sorted"
assert len(df) == N_WEEKS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(df['date'], df['sales'], linewidth=0.8)
axes[0, 0].set_title('Weekly Sales Over Time')
axes[0, 0].set_xlabel('Date')

axes[0, 1].hist(df['sales'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Sales Distribution')

# Monthly aggregation
df_monthly = df.set_index('date').resample('ME').sum()
axes[1, 0].bar(df_monthly.index, df_monthly['sales'], width=20, alpha=0.7)
axes[1, 0].set_title('Monthly Aggregated Sales')

# Autocorrelation
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df['sales'], ax=axes[1, 1])
axes[1, 1].set_title('Autocorrelation')
axes[1, 1].set_xlim(0, 60)

plt.tight_layout()
plt.savefig('eda_weekly_sales.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA plots saved.")

EDA plots saved.


## Target Analysis

In [7]:
print("Sales Statistics:")
print(df['sales'].describe())
print(f"\nCoefficient of Variation: {df['sales'].std() / df['sales'].mean():.3f}")
print(f"Skewness: {df['sales'].skew():.3f}")
print(f"Kurtosis: {df['sales'].kurtosis():.3f}")

Sales Statistics:
count       160.000000
mean      66951.667585
std       11256.536381
min       43753.455089
25%       58262.746709
50%       66652.359946
75%       74770.978322
max      101797.323533
Name: sales, dtype: float64

Coefficient of Variation: 0.168
Skewness: 0.292
Kurtosis: -0.274


## Train / Validation / Test Split Strategy

We use a **time-aware split** — no shuffling:
- **Train**: all weeks except last 16
- **Validation**: next 8 weeks
- **Test**: final 8 weeks

In [8]:
train = df.iloc[:-(VAL_WEEKS + TEST_WEEKS)].copy()
val = df.iloc[-(VAL_WEEKS + TEST_WEEKS):-(TEST_WEEKS)].copy()
test = df.iloc[-(TEST_WEEKS):].copy()

print(f"Train: {len(train)} weeks ({train['date'].min()} to {train['date'].max()})")
print(f"Val:   {len(val)} weeks ({val['date'].min()} to {val['date'].max()})")
print(f"Test:  {len(test)} weeks ({test['date'].min()} to {test['date'].max()})")

Train: 144 weeks (2021-01-04 00:00:00 to 2023-10-02 00:00:00)
Val:   8 weeks (2023-10-09 00:00:00 to 2023-11-27 00:00:00)
Test:  8 weeks (2023-12-04 00:00:00 to 2024-01-22 00:00:00)


## Preprocessing Strategy

Weekly data is already aggregated, so we do minimal preprocessing:
- Ensure sorted by date
- No scaling needed for tree-based models
- Features will be created in the next section

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print("Data sorted and ready for feature engineering.")

Data sorted and ready for feature engineering.


## Feature Engineering

We create lag features and rolling statistics for tabular ML approaches:
- **Lags**: 1, 4, 8, 13, 26, 52 weeks
- **Rolling**: mean and std over 4, 8, 13 week windows
- **Calendar**: week of year, month, quarter, year

In [10]:
def create_features(data):
    d = data.copy()
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    d['month'] = d['date'].dt.month
    d['quarter'] = d['date'].dt.quarter
    d['year'] = d['date'].dt.year

    for lag in [1, 4, 8, 13, 26, 52]:
        d[f'lag_{lag}'] = d['sales'].shift(lag)
    for w in [4, 8, 13]:
        d[f'rolling_mean_{w}'] = d['sales'].shift(1).rolling(w).mean()
        d[f'rolling_std_{w}'] = d['sales'].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full)
df_feat = df_feat.dropna().reset_index(drop=True)
print(f"Feature matrix shape: {df_feat.shape}")
print(f"Features: {[c for c in df_feat.columns if c not in ['date', 'sales']]}")

Feature matrix shape: (108, 18)
Features: ['week_of_year', 'month', 'quarter', 'year', 'lag_1', 'lag_4', 'lag_8', 'lag_13', 'lag_26', 'lag_52', 'rolling_mean_4', 'rolling_std_4', 'rolling_mean_8', 'rolling_std_8', 'rolling_mean_13', 'rolling_std_13']


## Baseline Model — Naive Forecasts

- **Naive (last value)**: predict last known sales value
- **Seasonal Naive**: predict the value from 52 weeks ago

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []

# Naive: last value
naive_pred = np.full(TEST_WEEKS, train['sales'].iloc[-1])
results.append(calc_metrics(test['sales'].values, naive_pred, "Naive (Last Value)"))

# Seasonal naive: value from 52 weeks ago
seasonal_source = df_full['sales'].values
test_start_idx = len(df_full) - TEST_WEEKS
seasonal_pred = []
for i in range(TEST_WEEKS):
    idx = test_start_idx + i - SEASON_LENGTH
    if idx >= 0:
        seasonal_pred.append(seasonal_source[idx])
    else:
        seasonal_pred.append(seasonal_source[test_start_idx + i - 1])
seasonal_pred = np.array(seasonal_pred)
results.append(calc_metrics(test['sales'].values, seasonal_pred, "Seasonal Naive (52w)"))

Naive (Last Value)             | MAE:    17332.3 | RMSE:    19217.0 | MAPE: 18.86%
Seasonal Naive (52w)           | MAE:    11005.0 | RMSE:    11740.5 | MAPE: 12.32%


## LazyPredict Benchmark (Lag-Feature Tabular)

We apply LazyPredict's `LazyRegressor` to the tabular (lag-feature) formulation
of the time series to quickly benchmark many regression models.

In [12]:
from lazypredict.Supervised import LazyRegressor

feature_cols = [c for c in df_feat.columns if c not in ['date', 'sales']]

# Time-aware split on the feature matrix
cutoff_test = len(df_feat) - TEST_WEEKS
cutoff_val = cutoff_test - VAL_WEEKS

X_train_lp = df_feat.iloc[:cutoff_val][feature_cols]
y_train_lp = df_feat.iloc[:cutoff_val]['sales']
X_val_lp = df_feat.iloc[cutoff_val:cutoff_test][feature_cols]
y_val_lp = df_feat.iloc[cutoff_val:cutoff_test]['sales']

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_train_lp, X_val_lp, y_train_lp, y_val_lp)
print("\nLazyPredict Top 10 Models (validation):")
print(lp_models.head(10).to_string())


LazyPredict Top 10 Models (validation):
                          Adjusted R-Squared   R-Squared          RMSE  Time Taken
Model                                                                             
MLPRegressor                      450.666030 -577.142039  72926.449545    0.100091
LinearSVR                         449.685841 -575.881795  72846.923040    0.010742
KernelRidge                       435.764905 -557.983449  71707.943908    0.022903
GaussianProcessRegressor          164.745904 -209.530448  44007.353615    0.023539
Lars                                7.204956   -6.977800   8566.617586    0.032470
ExtraTreeRegressor                  3.405495   -2.092779   5333.867906    0.009084
CatBoostRegressor                   2.589537   -1.043691   4335.861425    1.419750
KNeighborsRegressor                 2.539663   -0.979566   4267.296528    0.045656
NuSVR                               2.307689   -0.681314   3932.713114    0.011358
DecisionTreeRegressor               2.293417  

## FLAML AutoML Run

FLAML performs automatic model selection and hyperparameter tuning.
We exclude XGBoost due to a known compatibility issue with the current package version.

In [13]:
from flaml import AutoML

X_train_fl = df_feat.iloc[:cutoff_val][feature_cols]
y_train_fl = df_feat.iloc[:cutoff_val]['sales']
X_val_fl = df_feat.iloc[cutoff_val:cutoff_test][feature_cols]
y_val_fl = df_feat.iloc[cutoff_val:cutoff_test]['sales']

automl = AutoML()
automl.fit(
    X_train=X_train_fl, y_train=y_train_fl,
    task='regression',
    time_budget=30,
    metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED,
    verbose=0
)

flaml_val_pred = automl.predict(X_val_fl)
print(f"FLAML best model: {automl.best_estimator}")
results.append(calc_metrics(y_val_fl.values, flaml_val_pred, f"FLAML ({automl.best_estimator})"))

# Test set
X_test_fl = df_feat.iloc[cutoff_test:][feature_cols]
y_test_fl = df_feat.iloc[cutoff_test:]['sales']
flaml_test_pred = automl.predict(X_test_fl)
results.append(calc_metrics(y_test_fl.values, flaml_test_pred, f"FLAML Test ({automl.best_estimator})"))

FLAML best model: extra_tree
FLAML (extra_tree)             | MAE:     2584.4 | RMSE:     2786.8 | MAPE:  3.54%
FLAML Test (extra_tree)        | MAE:    10730.8 | RMSE:    13161.4 | MAPE: 11.44%


## StatsForecast — Statistical Forecasting

We use StatsForecast with AutoARIMA, AutoETS, and SeasonalNaive (season_length=52).

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', 'sales']].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'store1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

# Use train+val for fitting, predict test horizon
sf_train = sf_df.iloc[:-(TEST_WEEKS)]
sf_test_actual = sf_df.iloc[-(TEST_WEEKS):]

sf = StatsForecast(
    models=[
        AutoARIMA(season_length=SEASON_LENGTH),
        AutoETS(season_length=SEASON_LENGTH),
        SFSeasonalNaive(season_length=SEASON_LENGTH)
    ],
    freq='W-MON',
    n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_WEEKS)
sf_preds = sf_preds.reset_index()

y_test_sf = sf_test_actual['y'].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        pred = sf_preds[col].values
        results.append(calc_metrics(y_test_sf, pred, f"SF {col}"))

print("\nStatsForecast predictions complete.")

SF AutoARIMA                   | MAE:     2751.1 | RMSE:     4117.1 | MAPE:  3.18%
SF AutoETS                     | MAE:    12939.2 | RMSE:    15119.2 | MAPE: 13.91%
SF SeasonalNaive               | MAE:    11005.0 | RMSE:    11740.5 | MAPE: 12.32%

StatsForecast predictions complete.


## Top 3 Model Selection

We select the top 3 models based on test-set MAE from all approaches tested.

In [15]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('MAE').reset_index(drop=True)
print("\nAll Results (sorted by MAE):")
print(results_df.to_string(index=False))

top3 = results_df.head(3)
print("\nTop 3 Models:")
print(top3.to_string(index=False))


All Results (sorted by MAE):
                  model          MAE         RMSE      MAPE
     FLAML (extra_tree)  2584.414431  2786.846644  3.536662
           SF AutoARIMA  2751.114255  4117.075633  3.179336
FLAML Test (extra_tree) 10730.776501 13161.425001 11.437208
   Seasonal Naive (52w) 11004.982462 11740.493726 12.321269
       SF SeasonalNaive 11004.982462 11740.493726 12.321269
             SF AutoETS 12939.151264 15119.219245 13.909108
     Naive (Last Value) 17332.303093 19216.991718 18.858758

Top 3 Models:
                  model          MAE         RMSE      MAPE
     FLAML (extra_tree)  2584.414431  2786.846644  3.536662
           SF AutoARIMA  2751.114255  4117.075633  3.179336
FLAML Test (extra_tree) 10730.776501 13161.425001 11.437208


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test['sales'].values, 'ko-', label='Actual', markersize=4)

# Plot FLAML test predictions
ax.plot(test['date'].values, flaml_test_pred, 's--', label=f'FLAML ({automl.best_estimator})', markersize=4)

# Plot StatsForecast predictions
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', markersize=4)

ax.set_title('Top Models — Test Set Forecast vs Actual')
ax.set_xlabel('Date')
ax.set_ylabel('Weekly Sales ($)')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Forecast comparison plot saved.")

Forecast comparison plot saved.


## Error Analysis

In [17]:
errors = test['sales'].values - flaml_test_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML Predicted)')
axes[0].set_xlabel('Test Week')
axes[0].axhline(y=0, color='r', linestyle='--')

axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
axes[1].axvline(x=0, color='r', linestyle='--')

plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.1f}")
print(f"Std residual:  {np.std(errors):.1f}")

Mean residual: 10730.8
Std residual:  7620.6


## Interpretation and Business Insight

- **Trend**: Sales show a clear upward trend, suggesting business growth
- **Seasonality**: Strong yearly cycle with peak sales around Christmas/New Year
- **Forecast use**: 8-week forecasts allow monthly planning cycles
- **Model choice**: Tree-based models (via FLAML) capture the lag-feature patterns well;
  statistical models (AutoARIMA/ETS) capture the trend and seasonality directly

## Limitations

1. Synthetic data — real retail data has more complex patterns
2. Single store — multi-store forecasting would need hierarchical approaches
3. No external regressors (promotions, weather, economic indicators)
4. 52-week seasonality may be too rigid for real-world yearly patterns
5. Short forecast horizon (8 weeks) — longer horizons degrade accuracy

## How to Improve This Project

1. Use real retail datasets (e.g., Walmart, Rossmann from Kaggle)
2. Add external features: promotions, holidays calendar, weather
3. Try deep learning forecasters (N-BEATS, TFT via Darts)
4. Implement hierarchical forecasting for multi-store scenarios
5. Add prediction intervals / uncertainty quantification
6. Use Chronos-Bolt or TimesFM foundation models

## Production Considerations

- Retrain weekly with the latest data
- Monitor forecast accuracy with rolling MAPE
- Set up alerting for anomalous sales weeks
- Store forecasts in a database for downstream systems
- Use MLflow or similar for experiment tracking

## Common Mistakes

1. **Data leakage**: Using future data in lag features — always shift before rolling
2. **Shuffled splits**: Never shuffle time series — use chronological splits only
3. **Ignoring seasonality**: Weekly data has strong yearly cycles
4. **Over-fitting to recent trend**: Tree models can extrapolate poorly
5. **Not comparing to naive**: Always benchmark against simple baselines

## Mini Challenge / Exercises

1. Change the forecast horizon to 12 weeks — how does accuracy change?
2. Add a promotion feature and measure its impact on forecast accuracy
3. Try a 4-week rolling origin evaluation instead of a single test window
4. Implement a weighted ensemble of the top 3 models
5. Add prediction intervals using conformal prediction or quantile regression

## Final Summary / Key Takeaways

1. Weekly sales forecasting benefits from both statistical and ML approaches
2. Seasonal naive is a strong baseline for weekly data with yearly cycles
3. FLAML automates model selection effectively for tabular lag-feature approaches
4. StatsForecast provides interpretable statistical models with trend/seasonal decomposition
5. Always compare against naive baselines — they are often surprisingly competitive
6. 8-week horizons are practical for retail planning and achievable with moderate accuracy

In [18]:
import json
metrics = {
    'project': 'Weekly Store Sales Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to metrics.json")
print("\n=== Weekly Store Sales Forecasting — Complete ===")

Metrics saved to metrics.json

=== Weekly Store Sales Forecasting — Complete ===
